In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal


def normalize(samples, desired_rms = 0.2, eps = 1e-4):
  rms = np.maximum(eps, np.sqrt(np.mean(samples**2)))
  samples = samples * (desired_rms / rms)
  return samples

# WAVファイルの読み込み
sample_rate, samples = wavfile.read('/home/h-okano/DiffBinaural/FairPlay/binaural_audios_22050Hz/001265.wav')

# もしステレオの場合は、1チャネルに変換（例：平均値を使用）
if samples.ndim > 1:
    samples = samples.mean(axis=1)
print(sample_rate)
# スペクトログラムの計算
frequencies, times, Sxx = signal.spectrogram(samples, sample_rate)
# スペクトログラムのプロット（dBスケールに変換）
plt.figure(figsize=(10, 5))
plt.pcolormesh(times, frequencies, 10 * np.log10(Sxx), shading='gouraud')
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.title('Spectrogram')
plt.colorbar(label='Intensity [dB]')
plt.tight_layout()
plt.show()


In [1]:
import os
import numpy as np
import librosa

def convert_to_db(mag, eps=1e-10):
    return 20 * np.log10(np.maximum(mag, eps))

# 正規化関数（提供された関数そのまま）
def normalize(samples, desired_rms=0.1, eps=1e-4):
    rms = np.maximum(eps, np.sqrt(np.mean(samples**2)))
    samples = samples * (desired_rms / rms)
    return samples

# 対象ディレクトリ
audio_dir = '/home/h-okano/DiffBinaural/FairPlay/binaural_audios_22050Hz'

# グローバルなサンプル値の最大・最小値を初期化
global_max_sample = -np.inf
global_min_sample = np.inf

mmax_sample = -np.inf
mmin_sample = np.inf

# グローバルなスペクトログラムの dB 値の最大・最小値を初期化
global_max_db = -np.inf
global_min_db = np.inf

# ディレクトリ内の全 WAV ファイルを処理
for filename in os.listdir(audio_dir):
    if not filename.lower().endswith('.wav'):
        continue
    filepath = os.path.join(audio_dir, filename)
    #print(f'Processing: {filepath}')
    
    # 音声ファイルの読み込み（librosa.loadはデフォルトでモノラルに変換します）
    samples, sr = librosa.load(filepath, sr=None, mono=False)
    
    file_max = np.max(samples)
    file_min = np.min(samples)
    mmax_sample = max(file_max, mmax_sample)
    mmin_sample = min(file_min, mmin_sample)
    
    # 正規化
    samples_norm = samples#normalize(samples)
    
    # 正規化後のサンプル値の最大・最小を更新
    file_max = np.max(samples_norm)
    file_min = np.min(samples_norm)
    global_max_sample = max(global_max_sample, file_max)
    global_min_sample = min(global_min_sample, file_min)
    
    # スペクトログラムの計算
    stft = librosa.stft(samples_norm, n_fft=1024, hop_length=256, win_length=1024)
    magnitude = np.abs(stft)
    
    # dB スケールに変換
    # ref=np.max とすると各ファイル内で最大値が 0 dB となります。
    spectrogram_db = np.log1p(magnitude)
    
    # スペクトログラムの dB 値の最大・最小を更新
    spec_max = np.max(spectrogram_db)
    spec_min = np.min(spectrogram_db)
    global_max_db = max(global_max_db, spec_max)
    global_min_db = min(global_min_db, spec_min)

# 結果の表示
print(mmax_sample, mmin_sample)
print("正規化後の全ファイルにおけるサンプル値:")
print("  最大値:", global_max_sample)
print("  最小値:", global_min_sample)
print("スペクトログラム（dB スケール）の全ファイルにおける値:")
print("  最大値:", global_max_db)  # 各ファイルで ref=np.max を使っている場合、ほぼ0 dBになるはず
print("  最小値:", global_min_db)


0.9999695 -1.0
正規化後の全ファイルにおけるサンプル値:
  最大値: 0.9999695
  最小値: -1.0
スペクトログラム（dB スケール）の全ファイルにおける値:
  最大値: 5.6441107
  最小値: 1.0626611e-11


In [ ]:
!conda install -n env1 ipykernel --update-deps --force-reinstall

: 

In [10]:
import random
import os
import csv
import numpy as np
import torch
import torch.utils.data as torchdata
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import librosa
from PIL import Image
import soundfile as sf
from librosa.filters import mel as librosa_mel_fn
from utils.helpers import convert_to_db

def mel_spectrogram(y, n_fft, num_mels, sampling_rate, hop_size, win_size):
    # mel_basis と hann_window をキャッシュから取得
    mel = librosa_mel_fn(sr=sampling_rate, n_fft=n_fft, n_mels=num_mels)
    
    hann_window = torch.hann_window(win_size)
    # STFTを計算する
    spec = torch.stft(y, n_fft, hop_length=hop_size, win_length=win_size, window=hann_window,
                      center=True, pad_mode='reflect', normalized=False, onesided=True, return_complex=True)
    # 複素数の絶対値を計算する
    spec = torch.abs(spec)
    # メルスペクトログラムを計算する
    mel_spec = torch.matmul(torch.FloatTensor(mel), spec)
    
    mel_spec = convert_to_db(mel_spec)
    return mel_spec

x = torch.zeros((20224,))
y = mel_spectrogram(x, 1024, 80, 22050, 256, 1024)
print(y.shape)

torch.Size([80, 80])
